# sidekick juan: practice

This module/project pack is extracted from the local files listed below. Functions and line numbers are source locations; the original program is not executed by practice.

Use this with [the walkthrough](walkthrough.md).

**Source:**

- [4_langgraph/community_contributions/sidekick_juan/README.md](../../../../../../4_langgraph/community_contributions/sidekick_juan/README.md)

**Run one cell at a time:** predict → run with Shift+Enter → explain → change one thing. Select the root project .venv kernel.

**Offline adaptation:** code below uses fictional data or local libraries. It never executes the linked source, reads a credential file, sends messages, starts a UI/server, or changes the course databases. Simulations are labeled; they do not verify external integrations.

## 1. A guardrail makes a stop/continue decision

An input guardrail checks a request before later work. The tripwire is a boolean. This deterministic example teaches the branch; it is not the model-based name detector in the source.

**Predict:** Does Alice trigger this demonstration guardrail?

In [ ]:
from pydantic import BaseModel
class NameCheckOutput(BaseModel):
    is_name_in_message: bool
    name: str
def check_name(message):
    found = "Alice" in message  # Deliberately limited teaching rule.
    return NameCheckOutput(is_name_in_message=found, name="Alice" if found else "")
for request in ["Send from Alice", "Send from Head of Business Development"]:
    result = check_name(request)
    tripwire_triggered = result.is_name_in_message
    print("stop" if tripwire_triggered else "continue")
assert check_name("Send from Alice").is_name_in_message

**Check:** Yes for Alice, no for the title. A boolean decision does not establish that every personal name is detected.

**Change one thing:** Try another name and observe the limitation. Describe how you would evaluate false positives and missed names.

## 2. Reducer vs. checkpoints vs. conversation identity

A reducer combines state updates. A checkpointer retains state across invocations. A thread_id selects a conversation. This uses SQLite in memory, so it creates no database file.

**Predict:** Does a different thread_id share the same list?

In [ ]:
from typing import TypedDict, Annotated
from operator import add
from langgraph.graph import StateGraph, START, END
from langgraph.checkpoint.sqlite import SqliteSaver
class MemoryState(TypedDict):
    turns: Annotated[list[str], add]
builder = StateGraph(MemoryState)
builder.add_node("reply", lambda state: {"turns": ["assistant reply"]})
builder.add_edge(START, "reply")
builder.add_edge("reply", END)
with SqliteSaver.from_conn_string(":memory:") as saver:
    graph = builder.compile(checkpointer=saver)
    config = {"configurable": {"thread_id": "practice-1"}}
    graph.invoke({"turns": ["first user turn"]}, config)
    result = graph.invoke({"turns": ["second user turn"]}, config)
    separate = graph.invoke({"turns": ["new user"]}, {"configurable": {"thread_id": "practice-2"}})
    print(result["turns"], separate["turns"])
    assert len(result["turns"]) == 4 and len(separate["turns"]) == 2

**Check:** No. Same ID continues a thread; a different ID selects another. In-memory SQLite here disappears when the connection closes.

**Change one thing:** Replace :memory: with a deliberately chosen practice database only if you want persistence. Explain when it would survive process exit.

## 3. Retrieve evidence before answering

A retrieval workflow selects relevant chunks and passes them as context. This tiny keyword example illustrates the boundary without embeddings, a vector store, or a hosted model.

**Predict:** Which chunk matches the query about Python?

In [ ]:
chunks = [{"id": "a", "text": "Alex writes Python automation."},
          {"id": "b", "text": "Alex enjoys hiking."}]
query = "Python"
evidence = [chunk for chunk in chunks if query.lower() in chunk["text"].lower()]
answer_input = {"question": query, "evidence": evidence}
print(answer_input)
assert [chunk["id"] for chunk in evidence] == ["a"]

**Check:** Chunk a. This keyword selection does not implement the source vector/embedding integrations.

**Change one thing:** Try a query with no matches. Return an explicit lack-of-evidence result instead of inventing an answer.

## Rebuild from memory

1. Recreate the main data-flow step in a new scratch cell.
2. Add one case that should fail and identify its exception/stop condition.
3. Compare your version with the linked source cells or functions.
4. Explain where a real model or service would enter, and what it could change.

For deliberate original-source execution, see the shared [configuration guide](../../../../configuration.md). All provider/transport integrations remain UNVERIFIED at runtime.